In [ ]:
import os
os.chdir("..")

In [ ]:
from st_repl import SpatialReg
import pandas as pd
import geopandas as gpd
import polars as pl
from pathlib import Path
import tempfile
from jp_qcew import CleanQCEW

from jp_tools import download
sr = SpatialReg()
cq = CleanQCEW("data/")

In [ ]:
sr.quasi_data()

In [ ]:
df_qcew = sr.conn.execute(
            f"""
            SELECT
                year,
                qtr,
                phys_addr_5_zip,
                phys_addr_city,
                ui_addr_5_zip,
                mail_addr_5_zip,
                ein,
                first_month_employment,
                total_wages,
                second_month_employment,
                third_month_employment,
                naics_code
            FROM '{sr.saving_dir}processed/pr-qcew-*.parquet';
            """
        ).pl()
df_qcew = df_qcew.with_columns(
            first_month_employment=pl.col("first_month_employment").fill_null(
                strategy="zero"
            ),
            second_month_employment=pl.col("second_month_employment").fill_null(
                strategy="zero"
            ),
            third_month_employment=pl.col("third_month_employment").fill_null(
                strategy="zero"
            ),
            total_wages=pl.col("total_wages").fill_null(strategy="zero"),
        )
df_qcew = df_qcew.with_columns(
            total_employment=(
                pl.col("first_month_employment")
                + pl.col("second_month_employment")
                + pl.col("third_month_employment")
            )
            / 3
        )
df_qcew = df_qcew.filter(
            (pl.col("total_employment") != 0) & (pl.col("total_wages") != 0)
        )

df_qcew = df_qcew.filter(
            (pl.col("phys_addr_city") != "") & (pl.col("naics_code") != "")
        )
df_qcew = df_qcew.with_columns(pl.col("phys_addr_city").str.to_lowercase())      
df_qcew = df_qcew.group_by(["year", "qtr", "phys_addr_city"]).agg(pl.col("total_employment").sum())
df_qcew = df_qcew.rename({"phys_addr_city":"name"})
df_qcew = df_qcew.to_pandas()
df_qcew[df_qcew["name"].str.contains("mayaguez")]

In [ ]:
gdf = sr.county_geom()
import string

# Create a translation table to remove accents
accented_vowels = "áéíóúüñÁÉÍÓÚ"
unaccented_vowels = "aeiouunAEIOU"

translation_table = str.maketrans(accented_vowels, unaccented_vowels)

# Apply the translation table to remove accents from the 'name' column
gdf["name"] = gdf["name"].str.lower().str.translate(translation_table)
gdf["name"].to_csv("test.csv")

In [ ]:
master = gpd.GeoDataFrame(pd.merge(gdf, df_qcew, on="name"), geometry="geometry")
master.plot()

In [ ]:
master